# Sedov-Taylor Blastwave (2D)

The same case as `sedov_1d.ipynb`, one dimension up -- `warpSPH.cases.sedov`'s
sampler, stopping rule and initial conditions are dimension-generic, so most
of what that notebook's intro says applies here unchanged; this one only
covers what is different.

**How the profile panels read at `dim>1`.** Every particle is scattered
against its own distance from the origin $r = |x|$ (unsigned), and a vector
quantity (velocity) is read as its magnitude rather than one signed
component -- `dim==1`'s raw signed `x` and single velocity component stop
being the right invariant coordinate once the blast is genuinely radial
rather than 1D. This is the same "collapse, don't average" idea
`PORTING_EXAMPLES.md` describes for Sod's `x`, just with the coordinate a
radially symmetric problem actually has: the vertical spread at a given $r$
is the departure from perfect spherical symmetry. The two reference targets
(the full self-similar solve and the closed-form `beta`-fit estimate) are
unchanged from 1D, just no longer mirrored to negative `x`.

**Cost.** `nx=200` is 40,000 particles (`nx**dim`) -- the old
`07-Sedov_Taylor_Blastwave_2D.ipynb`'s resolution.

Like `sedov_1d.ipynb`, this notebook is meant to be **edited while it runs**:
the step loop stays unrolled in a cell rather than hidden inside
`warpSPH.runner.run()`. Plotting calls `drawSedov` directly rather than going
through `sedovCase.setupPlot`/`updatePlot` -- see `01-sod/sod_1d.ipynb`'s
intro for why that path does not live-update inside a Jupyter cell in this
environment.

Precision note: switching between single and double precision is controlled
in the import/configuration cell below, and requires a kernel restart to take
effect.

![](outputs/06-Sedov_Taylor_Blastwave_2D.gif)

In [ ]:
%matplotlib widget
from warpSPHBootstrap import bootstrap
rt = bootstrap(precision='float32', verbose=True)

from warpSPH import *
from warpSPH.cases.sedov import sedovCase, drawSedov
from warpSPH.runner import CaseSpec, buildContext, encodeFrames
from warpSPH.runner.display import figureOf, resolvePlotBackend, visualizeWithFallback
from warpSPH.io import exportSimulationSystem, prepExport

import os
import torch
from tqdm.autonotebook import tqdm

In [ ]:
# Every knob you'd otherwise reach for as a `--flag` on `sedov_2d.py`, made
# explicit and editable here. `sedovCase.defaults`/`.params` are the same
# values the CLI script starts from -- anything not overridden below just
# keeps its case default.
spec = CaseSpec(caseName=sedovCase.name, scheme=sedovCase.scheme, params=dict(sedovCase.params)) \
    .merged(**sedovCase.defaults)

spec = spec.merged(
    # --- discretisation ----------------------------------------------------
    nx=200,                    # particles across the domain in every dimension
    dim=2,
    L=2.0,

    # --- output --------------------------------------------------------
    plot=True, show=True, plotInterval=25,
    store=True, storeInterval=500,     # states mode -- one HDF5 file per stored step
    caseName='06-sedovTaylorBlastwave2D',

    # --- Sedov's own knobs ---------------------------------------------------
    params=dict(
        gamma=5 / 3, rho0=1.0, E0=1.0,
        goalRadius=0.8,             # tLimit is derived from this, not set directly
        initialization='hat',       # 'hat' | 'singular' | 'quadrant' -- see sedov_1d.ipynb
        viscositySwitch='NoneSwitch',
        adaptiveSupportScheme='Owen',
        adaptiveSupportCorrections=False,
    ),
)
spec

In [ ]:
# Initial-condition generation: explicit, using the real case code
# (`sedovCase.buildSystem` -> `buildSedov`), not re-derived here.
# `buildSystem` also replaces `ctx.spec.tLimit` with the analytic time to
# reach `goalRadius` -- read it back from `ctx.spec`, not the local `spec`.
ctx = buildContext(sedovCase, spec)
sedovCase.configureScheme(ctx)
system = sedovCase.buildSystem(ctx)
runningState = system.initializeNewState()

print(f'goalRadius = {ctx.param("goalRadius")}, goalTime = {ctx.spec.tLimit:.4g}')
print(f'{runningState.state.positions.shape[0]} particles, dt = {float(ctx.config.dt):.4g}')

In [ ]:
# Export/plot setup via the same generic hooks `warpSPH.runner.run()` uses
# internally -- nothing here is re-derived, only called explicitly.
ctx.exportPath = prepExport(spec.caseName, ctx.config, ctx.schemeConfig, ctx.scheme, ctx.exportFunction)
spec.save(os.path.join(ctx.exportPath, 'caseSpec.json'))
print(f'exporting to {ctx.exportPath}')

extraData = sedovCase.extraData(ctx, runningState)

# Direct drawSedov + plt.subplots(), not sedovCase.setupPlot -- see
# sedov_1d.ipynb's intro for why. `drawSedov` clears and redraws every axis
# itself, so this same call is used for both the first frame and every
# update below.
fig = axis = None
if spec.plot:
    ctx.imagePath = os.path.join(ctx.exportPath, 'images')
    os.makedirs(ctx.imagePath, exist_ok=True)
    import matplotlib.pyplot as plt
    fig, axis = plt.subplots(2, 2, figsize=(9, 6), squeeze=False)
    drawSedov(ctx, runningState, (fig, axis))
    fig.savefig(os.path.join(ctx.imagePath, 'frame_00000.png'))

if spec.store:
    exportSimulationSystem(ctx.exportPath, 'initialState', ctx.scheme, runningState,
                           exportAdjacency=False, stages=None, exportStagesAdjacency=False,
                           extraData=dict(extraData, frame_num=0))

In [ ]:
# The step loop, visible and editable. This is the same call
# `warpSPH.runner.runner._run` makes internally, unrolled here so a
# perturbation or an extra diagnostic can be injected directly around it.
dt = ctx.config.dt if isinstance(ctx.config.dt, float) else ctx.config.dt.cpu().item()
nSteps = int(ctx.spec.tLimit / dt)

trajectory = []
for i in (tq := tqdm(range(nSteps), leave=True)):
    # <-- hook point -------------------------------------------------------
    stepResult = ctx.integrator.function(
        state=runningState, f=ctx.stepFunction, dt=ctx.config.dt,
        config=ctx.config, schemeConfig=ctx.schemeConfig, verbose=False,
    )
    runningState = stepResult.state
    # -----------------------------------------------------------------------

    tScalar = runningState.t.item() if torch.is_tensor(runningState.t) else runningState.t
    row = sedovCase.diagnostics(ctx, runningState)
    trajectory.append(dict(row, step=i, t=tScalar))
    tq.set_description(f"t: {tScalar:.4f}, " + ", ".join(f"{k}: {v:.4f}" for k, v in row.items()))

    if fig is not None and (i % spec.plotInterval == 0 or i == nSteps - 1):
        drawSedov(ctx, runningState, (fig, axis))
        fig.canvas.draw()
        fig.canvas.flush_events()
        fig.savefig(os.path.join(ctx.imagePath, f'frame_{i:05d}.png'))

    if spec.store and (i % spec.storeInterval == 0 or i == nSteps - 1):
        exportSimulationSystem(ctx.exportPath, f'state_{i:04d}', ctx.scheme, runningState,
                               exportAdjacency=False, stages=stepResult.stages,
                               exportStagesAdjacency=True,
                               extraData=dict(extraData, frame_num=i))

In [ ]:
if spec.plot:
    encodeFrames(ctx.imagePath, ctx.exportPath)

## A spatial picture

The profile panels above collapse the angular direction away on purpose (that
collapse is what makes the vertical spread a symmetry-breaking measurement).
This cell instead draws the actual 2D density field with the particle
visualizer the field-map examples use (`warpSPHPlotting.visualize`, through
the runner's backend-with-fallback helper) -- a bowed or lopsided front here
is the same error the profile scatter reports as vertical spread, but located
in space.

In [ ]:
from warpSPHPlotting import PlottingOptions, UniformColorMap

plotter = visualizeWithFallback(
    ctx, resolvePlotBackend(ctx),
    particleState=runningState.state,
    domain=ctx.config.domain,
    quantities={'A': runningState.state.densities},
    # A uniform map, not the diverging one the field-map examples reach for:
    # density here runs monotonically from the ambient value to the shocked
    # peak and has no meaningful midpoint to diverge about.
    plotOptions={'A': PlottingOptions(colorMap=UniformColorMap.viridis, markerSize=3,
                                      plotTitle='density', vMin=ctx.param('rho0') * 0.9)},
    figTitle=f'{sedovCase.name}  t = {float(runningState.t):.4g}  '
             f'({runningState.state.positions.shape[0]} particles)',
    mosaic='A', figsize=(6, 5),
)
plotter.export(os.path.join(ctx.imagePath, 'field.png'), dpi=150)
figureOf(plotter)